<a href="https://colab.research.google.com/github/p36202600024-coder/S2-AI-RAG/blob/main/S2AI_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Menghubungkan Google Drive dan Mengakses Folder

Silakan jalankan kode di bawah ini untuk melakukan proses autentikasi dan menghubungkan Google Drive Anda ke notebook ini.

In [1]:
# @title
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# ID Folder dari link: https://drive.google.com/drive/folders/1JlQkXLfXCjTNbLdVaJXppnYjLNRKFMXU
folder_id = '1JlQkXLfXCjTNbLdVaJXppnYjLNRKFMXU'

# Menampilkan daftar file menggunakan Google Drive API atau mencari lokasi shortcut
print("Google Drive berhasil terhubung! Jika folder tersebut berada di 'Shared with me' (Dibagikan dengan saya), Anda disarankan untuk membuat pintasan (shortcut) folder tersebut ke 'My Drive' (Drive Saya) agar dapat diakses dengan mudah menggunakan path sistem berkas.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive berhasil terhubung! Jika folder tersebut berada di 'Shared with me' (Dibagikan dengan saya), Anda disarankan untuk membuat pintasan (shortcut) folder tersebut ke 'My Drive' (Drive Saya) agar dapat diakses dengan mudah menggunakan path sistem berkas.


In [2]:
!pip install -q langchain langchain-community langchain-text-splitters langchain-ollama faiss-cpu sentence-transformers pypdf

import os
import time
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

# ==============================================================================
# PROPOSED METHOD: LIGHTWEIGHT AGENTIC MULTIMODAL-RAG FOR CYBER PATROL
# Author: Nur Fauzi Pratama Nugraha (NIM: P36.2026.00024)
# Repo: https://github.com/p36202600024-coder/S2-AI-RAG
# ==============================================================================

def initialize_vectorstore(pdf_folder="./papers"):
    print("[1/3] Loading Documents & Splitting Chunks...")
    if not os.path.exists(pdf_folder):
        os.makedirs(pdf_folder)
        return None

    loader = PyPDFDirectoryLoader(pdf_folder)
    documents = loader.load()

    if len(documents) == 0:
        return None

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
    docs = text_splitter.split_documents(documents)

    print("[2/3] Generating Local Embeddings (HuggingFace)...")
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vectorstore = FAISS.from_documents(docs, embeddings)
    return vectorstore

def run_agentic_rag(vectorstore):
    print("[3/3] Executing RAG Pipeline with Llama-3...")
    retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

    llm = ChatOllama(model="llama3", temperature=0.1)

    prompt_template = """
    Anda adalah analis intelijen siber senior. Berdasarkan dokumen ilmiah berikut:
    {context}

    Tugas: Hasilkan Literature GAP Matrix dan evaluasi keterbatasan metode pada topik: "{question}".
    Sajikan dalam tabel Markdown dengan kolom: Penulis, Judul, Metode, Domain, Temuan, dan Gap.
    """

    prompt = PromptTemplate(template=prompt_template, input_variables=["context", "question"])
    qa_chain = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=retriever, chain_type_kwargs={"prompt": prompt})

    start_time = time.time()
    result = qa_chain.invoke({"query": "Sintesiskan GAP Matrix dari paper patroli siber dan radikalisme."})
    elapsed_time = time.time() - start_time

    print(f"[+] Processing Completed in {elapsed_time:.2f} seconds.")
    return result["result"]

if __name__ == "__main__":
    vdb = initialize_vectorstore("./papers")
    if vdb:
        output = run_agentic_rag(vdb)
        with open("Literature_GAP_Matrix.md", "w", encoding="utf-8") as f:
            f.write(output)
        print("[+] Output saved to Literature_GAP_Matrix.md")

/tmp/ipykernel_24062/1979149344.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader


ModuleNotFoundError: No module named 'langchain.chains'

In [1]:
# @title Cek Impor Pustaka (Jalankan setelah restart session)
try:
    from langchain.chains import RetrievalQA
    from langchain.prompts import PromptTemplate
    from langchain_community.document_loaders import PyPDFDirectoryLoader
    from langchain_community.embeddings import HuggingFaceEmbeddings
    print("✅ Semua modul LangChain berhasil diimpor dengan sukses!")
except ImportError as e:
    print(f"❌ Gagal mengimpor modul: {e}")
    print("Silakan pilih menu 'Runtime' -> 'Restart session', lalu jalankan sel instalasi kembali.")

❌ Gagal mengimpor modul: No module named 'langchain.chains'
Silakan pilih menu 'Runtime' -> 'Restart session', lalu jalankan sel instalasi kembali.


In [2]:
!pip list | grep -E "langchain|pypdf|faiss|sentence-transformers"

faiss-cpu                             1.15.1
langchain                             1.4.7
langchain-classic                     1.0.8
langchain-community                   0.4.2
langchain-core                        1.7.0
langchain-ollama                      1.1.0
langchain-protocol                    0.0.20
langchain-text-splitters              1.1.3
pypdf                                 6.21.2
sentence-transformers                 6.2.0


### Alternatif Menggunakan PyDrive2

Jika folder tersebut dibagikan secara publik atau Anda ingin melihat daftar filenya langsung melalui API tanpa membuat pintasan manual di Drive Saya, Anda bisa menggunakan kode di bawah ini:

In [ ]:
# @title
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

# Autentikasi pengguna
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive_client = GoogleDrive(gauth)

# List berkas di dalam folder
file_list = drive_client.ListFile({'q': f"'{folder_id}' in parents and trashed=false"}).GetList()

print(f"Menemukan {len(file_list)} file di dalam folder:")
for file in file_list:
    print(f"- Nama: {file['title']} | ID: {file['id']}")